Attempting similar hard coded right hand side, but with the starting function u=21-4x+3y

In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and right and default=Neumann on the other 2 sides
fes = H1(mesh, order=1, dirichlet="left|right")
fes.ndof

# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()


In [ ]:
# Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

# Generating boundary values for Dirichlet boundaries and RHS from u=21-4x+3y 
# then that's what we are hoping to see upon solve
plant = CoefficientFunction(21 - 4 * x + 3 * y)

# Right hand side
f = LinearForm(fes)
# generated f from a known input u=1-3x^2+2x^3 then we hope to solve for it
#fakef = 25/4 -12*x + 12*x*y - 12*x*y*y + 69/4 *x*x - 12*x*x*y - 47/2 *x*x*x + 6*y*y
# generated f from  f=-div(k*grad(plant))+sigma^2*plant
fant = 7 * x - 21/4 * y + 21/4
f += fant*phi*dx
f.Assemble()

# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(plant, BND) # this sets the Dirichlet boundary component to 1, might want to modify later

print(a.mat)
# print(f.vec)

In [ ]:
# To see the K(x,y) function, we plot it on our mesh
# Draw(Kxy, mesh)
# Draw the initial state of the grid function 
# Draw(gfu)

# Solve the PDE
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu)

In [ ]:
Draw(plant,mesh)